<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/cross_app_authz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta Cross-App Access Agent-to-Agent Demo

This notebook demonstrates the complete **Identity Assertion Authorization Grant (ID-JAG)** flow for secure cross-application access using the Okta AI SDK.

For this notebook we will be working with the following resources:

| Entity | Identifier | Role
|---|---|---|
|User| Your user | Authenticates to App |
|App| n/a | Sends user access token to agent in API call |
|Agent J| https://api.atko.rocks/agent/j | 1. Exchanges user's access token for an ID-JAG targeting Agent B's Auth Server. <br>2. Makes API call to Agent B. |
|Agent K| https://api.atko.rocks/agent/k | 1. Exchanges Agent A's access token for an ID-JAG targeting the Resource Auth Server. <br>2. Makes tool call to resource. |
|Resource| api://mcp | |

### The Sequence / Steps

1. **Get User Access Token:** Authenticate as a user to obtain an access token.
2. **Exchange Access Token for ID-JAG Token:** Convert user's Access token to a JWT Assertion Grant Token targeting *Agent J*.
   - This is handled by the IDP's Auth Server -- the Okta Org Authorization Server in this case.
3. **Verify ID-JAG Token:** Validate the token's authenticity and claims. 
   - This step is *technically* optional but we recommend it for this exercise.
4. **Exchange ID-JAG for new Access Token:** Get an access token for access to Agent K.
5. **Call Agent K / Validate Access Token:** Just like with any other API, **Agent K** should validate the access token before use.
6. **Exchange Access Token for ID-JAG Token:** Convert Agent K's Access token to a JWT Assertion Grant Token targeting the *resource*.
   - This is handled by the IDP's Auth Server -- the Okta Org Authorization Server in this case.
7. **Verify ID-JAG Token:** Validate the token's authenticity and claims. 
   - This step is *technically* optional but we recommend it for this exercise.
8. **Exchange ID-JAG for new Access Token:** Agent K requests an access token for access to the *resource* by cashing in the ID-JAG.
9.  **Call Resource / Validate Access Token:** While this notebook does _not_ actually call an API it would be standard practice for the resource to validate this access token.

> Training intent: this notebook prioritizes conceptual clarity and runnable sample code for ID-JAG.

[![](https://mermaid.ink/img/pako:eNqdVttuo0gQ_ZVS78usgjOm8RWtIiGyGmVGGa98ycPKL23cJiiY9jbNTCZR_n2raGxwwDPateUL1Ok6dQ5HDa8sUlvJfJbLfwqZRfI2EbEW-3UG-BKFUVmx30htjw9CmyRKDiIzsAKRwyo_luz3Rj3Dcr5aLOF2dh_cff1jo2_oE6aJxDUfIYjp9zOda7cMqWUFDQ6HDkCwIEhQmMcXWEj9rWuwwC0xlsiWZbZtTtlEz2zL2ZMRHX1_rspyfOkWE_B6jC9d5Uta3g17iXwuc1XoSHazz6n3EdJR_ik5vo2WkQEdbz70HXD5xAE-HP5OhbX5qowEhUtg5UDow8LIA7h-2Q3VJpHAuk0Gwddm1bu5Cc_rNPW3RMBGq-8NZEjIYGGxSicvwiQqgxAzSiuuYKmeZAaawpqb06pg0bMUxAoiimSegyEoQdqa3DGq6ndrCh0oJyhVcR8-SQMig7vb3ufgUz0n8gWoeW4ngStLHVjq5ZGa3oGL2Bm1_PM5ehRZLNtYErdTumIBI3QsTZLFdX5sr5lVSswVdHlZJV254Qi_XN5UmcqdAbUDalKK9Hx4kDrZ_bAgdTBnoxPO1lucR9YO8imyD6bocpNaJ_HjOfeg4cq5xZabn_lW8ZNTlTGti10u5bVNwa_CIMinCdnEcVy3lYfAdWB2CsQQO_51B5FIU_j-sYO9GrtOhlGnWXFBOyBNu3m1trK7DT6zuyHBnXhloN0h79KA8ztAjUsNow7PoXqVI5yHtamRrKestnJabzXNmPJTTM3_j-lx7PHlmL7z7T_EFHcAa5w7bV957oANYMk_uRDVkn1-KadHY95F5ej3vHbql0n94OE2PMYL7fa7h51Xk05tSMPLIeVEOj8L6WnQq3e5O41qlzyINNnSDn9BkEWhD9tU1ht1Xa7UapkfVJbLSilzWKyTLfONLqTD9lLvBR2yV1q7Znjn2Ms18_GvltviubcV-qkXqVTpNXMshk7d452CYGWbxtIHoROxSWVOxdfq_so2InqKtSrQ6bL1b57nrRlV39bZG86EN8u_ldofx0Jo_Mj8nUhzPCoO5EP1wHQ6q1GN1CE2Ncx3RxPcBKkN81_ZM54Y9q8HI3c8Goxd3ufuwGE_mN_jfHo9mAyHvI_nvP5wOn5z2EvJ7F57Xt-bDiajQX_CPW-ES-Q2MUrf2ye38gHu7V_y0yPd?type=png)](https://mermaid.live/edit#pako:eNqdVl1vo0gQ_CutuZc9BWcNGIPRKRIip1V2lfXJH3k4-WWMxwQFM75hvJtNlP9-3Qw2OOBd3dmKHdPVXV1FCXhlidwIFrJS_HMQRSJuM54qvlsVgC9-0LI47NZCmd97rnSWZHteaFgCL2FZHkvmcy2fYTFbzhdwO72P7r7-sVY39BfnmcCejxCl9P2ZjnVHxjSyhkb7fQ8gmhMkOujHF5gL9a1vsciuMIbIlEWxaW_ZRk_NyOmT5j1zf67KcHzpFxM5zRpf-sqXtLxb9hL5TJTyoBLRzz6j2UdIT_mn5PjWSiQaVLr-MLTAdgILHM_7nQor_VVqARJbYGlBHMJciz3YYTUN1WYJx7pJBsFXejm4uYnP67T1t4zDWsnvLWRMyGhusFJlL1xnsoAYM0odV7CQT6IARWEt9akrmg8MBbECTxJRlqAJSpCuJttHVcN-TbEF1QaVKieET0IDL-DudvA5-tTsiXwRap6ZTeDKUEeGenGkpndkI3ZKI_98Th55kYoulsRtpapZQHOVCp0VaZMfM2tqlBJzDV1cVklnzhvjh-20VeZiq0FugYZUIt0QHoTKtj8MSO712eqEM_UO55G1h3yC7KMJutymVln6eM49arlybrHhds58q_nJqdqYzsmuWp3GpuhXYeDkU0A2Obiu3clDZFswPQXCw4l_3UHC8xy-f-xhr9dukqHlaVds6AakbbdT99Z2d8Fndrck2IFbBdr2nD4NuL8FNLjSMO7xHOpXtcJ5WNsayXrKaienzaWmHVPnFFP9_2N6XNu_HNN3vv2HmOIVwBhnT7pn3rHABLDiDy5EtWKfXcrp0Zh3UTn6PWuc-mVSP7h4GfbxRNvD_mVn9aYTE9L4ckgdIp2dhfS06NW73J1WNS0PPM82dIW_IMig0IdNLpoLdVOu1SpR7mVRilops1iqsg0LtToIi-2E2nH6yV6pd8XwzrETKxbiv0psDs-DDVdPg0TmUq2YZTB06B7vFASrxrRaH7jK-DoXJRVf6_srW_PkKVXygE5Xo39zXXfFqPq2Kt5wJ7xZ_i3l7rgWQtNHFm55XuKvw558qB-YTkcVqhEqxqGahfbYDybVGBa-smcWDmz_2nMDx3Y92_PdiR-MLPYDj7t-cD0KPDzq-J47Hg7HbxZ7qbjta9cdupNRMB4NA8d1x9giNpmW6t48u1WPcG__Ajw0JGQ)

---

### Prerequisites

Before running this notebook, you need:

1. **Okta Organization** with:
   - Custom authorization server configured
   - OAuth 2.0 application with Token Exchange enabled
   - ID-JAG support enabled

2. **Two Agents/Workload Principals** with:
   - Principal IDs (agent identifier)
   - Private JWKs (RSA key pair for JWT bearer assertion)

3. **User**:
   - Valid Okta User to authenticate and obtain an access token.

## Setup and Installation


> #### <br>**IMPORTANT**
> 
> **Training Notebook Note**
> - This notebook intentionally includes some sample-code shortcuts for readability and step-by-step learning.
> - For production: store secrets outside notebooks, enforce full token verification, and package helper logic into tested modules.
> - Before publishing to GitHub, remove local-only utilities and any environment-specific scaffolding.<br><br>

### 1. Install the Okta Python SDK

In [ ]:
# Install the Okta AI SDK from PyPI

%pip install --upgrade okta-client-python

### 2. Set Configuration

Set _all_ your configuration variables and _run the cell._

In [ ]:
# @title { display-mode: "form", vertical-output: true }
# @markdown _Enter your configuration variables here and click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to validate._
# @markdown <br><br> These will be used throughout the notebook.

from okta_client.oauth2auth import AuthorizationCodeFlow, CrossAppAccessFlow, CrossAppAccessTarget
from okta_client.authfoundation import OAuth2Client, Token
from typing import TypedDict
import utils
from pathlib import Path
import importlib
import sys
# @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
OKTA_DOMAIN = ''
# @markdown <br>

# @markdown ---
# @markdown ##### Domain A: Client / Agent configuration
# @markdown ---
# @param {"type":"string","placeholder":"Enter your authorization server Id (or full issuer URL)"}
DOMAIN_A_ISSUER = ''
# @markdown <br>
# @param {"type":"string","placeholder":"Enter application redirect URI"}
REDIRECT_URI = 'http://localhost:8080/authorization-code/callback'
# @markdown <br>
# @param {"type":"string","placeholder":"Enter your client Id"}
CLIENT_ID = ''
# @param {"type":"raw","placeholder": "Enter the scopes your client should request."}
CLIENT_SCOPES = ['openid', 'profile', 'email', 'mcp:read']

# @markdown <sup><em>If you are using client secret authentication, enter your client secret below.</em></sup>
# @param {"type":"string","placeholder":"Enter your client secret"}
CLIENT_SECRET = ''
# @markdown <sup><u>or...</u> <em>If you are using private key authentication, enter your application's private JWK below.</em></sup>
CLIENT_PRIVATE_JWK = {}  # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @param {"type":"string","placeholder":"Enter Agent A's identifier"}
PRINCIPAL_A_ID = ''
# @param {"type":"string","placeholder":"Enter the audience/resource URL configured for Agent A."}
PRINCIPAL_A_RESOURCE_URI = 'https://agent_a'
# @param {"type":"raw","placeholder": "Enter the scopes Agent A should request."}
PRINCIPAL_A_SCOPES = ['mcp:read']

# @markdown <sup><em>If you are using client secret authentication, enter Agent A's secret below.</em></sup>
# @param {"type":"string","placeholder":"Enter Agent A's client secret"}
PRINCIPAL_A_SECRET = ''
# @markdown <sup><u>or...</u> <em>If you are using private key authentication, enter Agent A's private JWK below.</em></sup>
PRINCIPAL_A_PRIVATE_JWK = {}  # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Domain B: Agent Configuration
# @markdown ---
# @markdown <sup><em>Only enter a value here if it is different from DOMAIN_A_ISSUER.</em></sup>
# @param {"type":"string","placeholder":"Enter your authorization server Id (or full issuer URL)."}
DOMAIN_B_ISSUER = ''
# @markdown <br>
# @param {"type":"string","placeholder":"Enter the audience/resource URL configured for Agent B."}
PRINCIPAL_B_RESOURCE_URI = 'https://agent_b'
# @markdown <br>
# @param {"type":"string","placeholder":"Enter Agent B's identifier"}
PRINCIPAL_B_ID = ''
# @param {"type":"raw","placeholder": "Enter the scopes your agent should request."}
PRINCIPAL_B_SCOPES = ['mcp:read']

# @markdown <sup><em>If you are using client secret authentication, enter Agent B's secret below.</em></sup>
# @param {"type":"string","placeholder":"Enter Agent B's client secret"}
PRINCIPAL_B_SECRET = ''
# @markdown <sup><u>or...</u> <em>If you are using private key authentication, enter Agent B's private JWK below.</em></sup>
PRINCIPAL_B_PRIVATE_JWK = {}  # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Resource Server Configuration
# @markdown ---
# @param {"type":"string","placeholder":"Enter an authorization server Id or the full authorization server issuer URL"}
RESOURCE_ISSUER = 'https://atko-ai.oktapreview.com/oauth2/ausubr9dq7o80RBml1d7'
# @markdown <br>
# @param {"type":"string","placeholder":"Enter your resource server audience"}
RESOURCE_SERVER_AUDIENCE = 'https://benefits.streamward.com'
# @markdown <br>

# @markdown ---
# @markdown <em>Only enable this if you are having issues and need to see a lot of logging.</em>
SDK_DEBUG_ENABLED = False  # @param {"type":"boolean"}

# =============== LOAD NOTEBOOK UTILITY FUNCTIONS ==============
# Prefers a local utils.py so edits are picked up without a kernel restart; otherwise fetches it into the working directory.

UTILS_URL = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/utils.py"

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "utils.py").exists()), None)
IS_LOCAL = repo_root is not None

if repo_root is None:
    import requests
    response = requests.get(UTILS_URL, timeout=15)
    response.raise_for_status()
    Path("utils.py").write_text(response.text)
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

importlib.reload(utils)

print(f"✅ Utilities loaded from {utils.__file__}")
# ==========================

try:
    # Issuer values are 'reset' here in order to account for variation between using an authorization server ID vs a full issuer URL. The get_issuer util function will normalize this.
    DOMAIN_A_ISSUER = utils.get_issuer(OKTA_DOMAIN, DOMAIN_A_ISSUER)
    # The issuer for principal/agent B if different from principal/agent A's issuer. This can be made configurable in the future.
    DOMAIN_B_ISSUER = utils.get_issuer(OKTA_DOMAIN, DOMAIN_B_ISSUER)
    RESOURCE_ISSUER = utils.get_issuer(OKTA_DOMAIN, RESOURCE_ISSUER)

    utils.validate_config(locals(), "a2a")
except ValueError as e:
    print(e)
    raise SystemExit(
        "❌ Configuration validation failed. Please fix the above errors and re-run the cell.")

utils.a2a_config_out(locals())

# The following is placed here in this cell as this cell is a form and hides the code. This helps the notebook stay cleaner and easier to follow.
# Create some types to make this notebook more stable


class ActorState(TypedDict, total=False):
    sdk: OAuth2Client | None
    auth_flow: AuthorizationCodeFlow | CrossAppAccessFlow | None
    access_token: str | None
    id_token: str | None
    refresh_token: str | None
    id_jag_token: Token | None
    # result: Token | None


class State(TypedDict):
    user: ActorState
    agent_j: ActorState
    agent_k: ActorState


# Initialize the global STATE dictionary with default values for user, agent_j, and agent_k.
STATE: State = globals().setdefault(
    "STATE",
    State({"user": ActorState(), "agent_j": ActorState(), "agent_k": ActorState()}),
)

### 3. Initialize the SDK for _user authentication_

In [ ]:
# Initialize Okta SDK
from okta_client.authfoundation import OAuth2Client, OAuth2ClientConfiguration, ClientSecretAuthorization, LocalKeyProvider
from okta_client.authfoundation.oauth2.jwt_bearer_claims import JWTBearerClaims
from okta_client.authfoundation.oauth2.client_authorization import ClientAssertionAuthorization
from okta_client.oauth2auth import AuthorizationCodeContext, AuthorizationCodeFlow, CrossAppAccessFlow, CrossAppAccessTarget, Prompt

# ============ HELPER FUNCTION(s) ==========
def build_client_authorization(client_id: str, client_secret: str, private_jwk: dict, audience: str = f"{OKTA_DOMAIN}/oauth2/v1/token", label: str = "okta_sdk"):
    """This is a helper function specifically for this notebook to showcase how you can implement the Python SDK. It chooses private key authentication if a valid private JWK is provided; otherwise, it falls back to client secret authentication."""

    if private_jwk and isinstance(private_jwk, dict) and private_jwk.get("kid"):
        print(
            f"\n✅ Using private key auth for {label} (kid={private_jwk.get('kid')})")
        return ClientAssertionAuthorization(
            assertion_claims=JWTBearerClaims(
                issuer=client_id,
                subject=client_id,
                audience=audience,
                expires_in=300
            ),
            key_provider=LocalKeyProvider(
                key=private_jwk,
                algorithm=private_jwk.get("alg", "RS256"),
                key_id=private_jwk.get("kid")
            )
        )

    print(f"\n⚠️ Using client secret auth for {label}.\n   We advise a more secure method in a production environment.")

    return ClientSecretAuthorization(
        id=client_id,
        secret=client_secret
    )
# ========== END OF HELPER FUNCTION(s) ==========

# Initialize the user SDK with the appropriate configuration, including the issuer for the custom authorization server.
user_sdk = OAuth2Client(
    configuration=OAuth2ClientConfiguration(
        issuer=DOMAIN_A_ISSUER,
        scope=CLIENT_SCOPES,
        redirect_uri=REDIRECT_URI,
        client_authorization=build_client_authorization(
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
            private_jwk=CLIENT_PRIVATE_JWK,
            label="client application"
        ),
        additional_parameters={
            # The `resource` parameter is used (per RFC8693) to specify the audience/resource for which the access token is requested. In this case, it is set to the RESOURCE_URI defined earlier.
            "resource": PRINCIPAL_A_RESOURCE_URI
        }
    )
)

STATE["user"]["sdk"] = user_sdk

print("✅ User SDK initialized!")

if SDK_DEBUG_ENABLED:
    user_sdk.listeners.add(utils.Debugger())  # type: ignore

---

## Step 1: Obtain User Tokens

### Important:
This step uses a **Custom Authorization Server**. 

This is the correct approach for obtaining the initial user token that will be used in the ID-JAG flow.

### The Sequence / Flow

1. **Authenticate via browser:** The client application redirects the user to an OIDC login page.
2. **Exchange auth code:** Okta will return an authorization code, which the client will send to Okta for an access token.
3. **User authenticated:** Okta returns the access, ID token, and optionally a refresh token (if `offline_access` is requested).

#### In this exercise we will...

1. **Build an authorization URL:** This will allow us to simulate the behavior of your client application.
2. **Copy redirect URL from browser:** Since we do not have an actual client to handle the redirect, manually copy the **entire** redirect URL back to the notebook.
3. **Exchange code for tokens:** Get user tokens where issuer = Client/Agent J's trust domain.

[![](https://mermaid.ink/img/pako:eNptUk2Pm0AM_SvW9NKqJCWQIYCqSIi9tFK2Ugk9VFwGmLKjBIaaYZsmyn_vGJLVVi2ID9vPz88eX1ila8liNsifo-wq-aBEg6ItOrCXGI3uxraUONu9QKMq1YvOQA5igHy4h-Z3qU-w_5pne3j4sks-PX4scUtPelTS5nyApKHvZ_L9S5kS5Q2a9P1_AElGkGQ0T2fIJD7fq8uuvsuYboOyMoBN-dZ1YOWFDnicv6NAYR61kaBtKuQOpDFkRvawiidWW1lVwsbnxghemHyx3aZ_x0n_sxJQov71CpkSMslmrEZ1FkbpDlI7Ysp4D3t9kB0gzXowL1lJtphLUFUQVSWHAQxBCTL1xhzWoKpZbHCUDmsltoJMdqG-C2altbJgsf1FWY-nRS3wsKj0UWPBnBlDrp2VQrCJ5lXqN4FKlEc5UPBymyUrRXVoUI9WwUT9xvf9glH0WnRXq8meynet27ssC22eWPxDHAdrjX1tR3VbqBcv2n4kppbUsHi1cXk00bD4wk7WEXjLlc-Dte_zcL1xA89hv6075Mso8r0ocCPur3m4uTrsPFV2l4EbRl7gcx764WbDA4fJWhmNu3mzpwW__gHDdO-B?type=png)](https://mermaid.live/edit#pako:eNptUtuOmzAQ_ZWR-9KqJOUSLkFVJMS-tFK20hL6sOLFgMtaCZgOZpsmyr_XA8lqV10QF885c-aMPWdWqVqwmA3i9yi6StxJ3iBviw7MxUeturEtBc7rnqOWlex5pyEHPkA-3KD5Xaoj7B7ybAd3P7bJt_uvJW7oSQ9SmJwvkDT0_U6x_yVTkrxSk75_h5BkRElG_XSCTODze8YSZ-LMhWZYdPXN5XRrFJUGbMqPtgWOG1ng-v4nAgp9r7QAZZQhtyCNIdOiByeeihpJWXGDz30TvdD5YrNJ3-LU3rPkUKL684qZEjPJZq5CeeJaqg5ScwKU8Rl2ai86QDqKQb9kJdliLkFVgVeVGAbQRCXK1BuzWIOyZrHGUVisFdhyWrIz9V0wY60VBYvNL4p6PC5qjvtFpQ4KC2bNHAptjRWiTTKvUn9ylLw8iIHA83UvWcmrfYNqNA4m6Q-e5xWM0EvRXYwncx6PSrU3W4baPLH4Fz8MZjX2tdmq67y9RNH0IzA1oprFTmj7ziTD4jM7mkDgLh3PD1ae50er0A5ci_014chfrteeuw7ste-t_Ci8WOw0VbaXgR2t3cAL3TB0HNc2GaKWWuF2Hvxp_i__AGoI-Ug)

### ① Build Authorization URL and Authenticate via the browser

Run the following cell and then click the generated button to open the authorization URL in a new browser tab.

In [ ]:
# Build the authorization URL
print("\n⏳ Building authorization URL...")

assert isinstance(sdk := STATE["user"].get("sdk"), OAuth2Client), "❌ Missing or invalid User SDK. Initialize the User SDK first."

auth_flow = AuthorizationCodeFlow(client=user_sdk)

# Update the state with the new authorization flow. We will need this later.
STATE["user"]["auth_flow"] = auth_flow

authorization_url = await auth_flow.start(context=AuthorizationCodeContext(
    prompt=Prompt.LOGIN,
))

print("✅ Authorization URL generated!")
print(f"\n{authorization_url}")
print("\n" + "="*80)

utils.render_button(authorization_url, REDIRECT_URI)

print("Note: The tokens will be issued by the Client Authorization Server")
print(f"      Issuer: {DOMAIN_A_ISSUER}")

### ② Exchange Authorization Code for Tokens

After authenticating...
1. copy the *entire* URL
   - *The entire URL is necessary as the SDK will validate state before initiating the exchange.*
2. paste the URL below
3. and run this cell to obtain tokens

In [ ]:
# @title { display-mode: "form" }

# @param { type: "string", placeholder: "Insert entire URL string here."}
REDIRECT_URL = 'http://localhost:8080/authorization-code/callback?code=m0vPLNRqdBM_evVyrtZljbqP5VBM5DoTAdLAlQoyvWs&state=72f8bd68-39f4-409c-86bf-38f5720ccee7'

try:
    if not REDIRECT_URL:
        raise SystemExit(
            "\n❌ No url provided! Please paste the entire URL containing the `code` and run this cell again.")

    print("\n" + "=" * 80)
    print("② Exchange Authorization Code for User Tokens")
    print("=" * 80)

    assert isinstance(flow := STATE["user"].get(
        "auth_flow"), AuthorizationCodeFlow)
    token = await flow.resume(REDIRECT_URL)

    # Extract tokens
    STATE["user"]["id_token"] = token.id_token.raw if token.id_token is not None else None
    STATE["user"]["access_token"] = token.access_token
    STATE["user"]["refresh_token"] = token.refresh_token

    utils.token_exchange_out(token, focus="access_token",
                             expected_issuer=DOMAIN_A_ISSUER)

except Exception as e:
    print(f" ❌ Error during token exchange: {e}")
    print("\nTroubleshooting:")
    print("   • Make sure you copied the entire authorization code")
    print("   • Verify your redirect_uri matches what's registered in Okta")
    print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)")
    print("   • Ensure your client_id and client_secret are correct")

---
## Step 2: Verify the Access Token

Next, the client would actually *use* the access token. Normally this validation would be done by the API being called. It is listed as optional in this workbook but, in a production environment, we would highly discourage this being optional!

### The Sequence / Flow:
4. Client App sends a request to Agent J using the newly minted access token in the `Authorization` header.
5. Agent J validates the token signature using the authorization server's public keys
    - Checks audience, issuer, and expiration
    - Extracts scope and user information

[![](https://mermaid.ink/img/pako:eNptUlFvmzAQ_iun28smkRQCIcSaIiH60knppjbrQ8WLMReCApgZo6WL8t9n42SbtCKZu_u-785n-84oZEnIcKAfI3WC7mteKd7mHZiPj1p2Y1uQgsgh7l_IE-yevj_v4P7rNn14_FyojV1ZU1On4Q7SytovFnMZPVe6FnXPDZwBH-AqTfv-f0EaWMW1hqOpK_POuYqEBlUV_GMSehAsYw98Yz8ZXuf6UWoCVVcHDXIPGYNnTT0sGKTfHkDwpoGfd8CFoGEALY_UubRsttmkAYMnew2DNtRtf6tPnX73Vy977Zxcp8E194VUvX97R-yatwY9rFRdItNqJA9bUi23IZ7t2XLUB2opR2ZcReV4mpVcHWdCNlLl6DmNhbbm0axsKvNP6gtXNS8aGix5vj4ZFlwcKyVH08ZU-kMYhjla9pJ3F9OTufVXKdtbW0ZaHZDteTOYaOxLrm9z8QdV5jSkMlNUIwuicCqC7IwnE66SeRLF0cpfG-NHvmHfkK3iuUWWi7UBk2CdXDz8NW3rzxM_DpZrs5ZBHMeL2EMqay3V1k3nNKSX3wp-2B4?type=png)](https://mermaid.live/edit#pako:eNptUlFvmzAQ_iun28smkRRCQsCaIqH0pZPSTW3Wh4kXA1eKApgZs6WL8t97xsk2aUMyd_fdd-fP9p2wUCWhwIG-j9QVdFvLSss264A_ORrVjW1OGpYOcf9cHWH_8PVxD7efd-nd_cdcb-zaNjV1Bm4graz9ZDFX0Utt6qLuJcNbkANcqGnf_0tIA8u49HBp6sqsc66mwoCucvk-Dj0IVpEHPtsPnDeZuVeGQP1gxVsBj4Z6WAhIv9xBIZsGft6ALAoaBjDqQJ0r2c42mzQQ8GCvYDCcuu5t-anj7__wVW-ck5k0uNQ-ka6fX_9DdsKtQQ8rXZcojB7Jw5Z0K22IJ3uuDM0LtZShYFdTOR5npdSHWaEapTP0HMdCO34wS5va_FX6JHUt84YGmzxdngtzWRwqrUaWMbV-F4ZhhjZ7zroza-Ib_6ZUe5XF1OoFxbNsBo7GvpTmOhO_Uc2nIb3lpgZFECZTExQnPHK4jufxMlqu_YSNv_RDD19RrKO5RVaLhME4SOKzh7-mbf157EfBKuG1CqIoWkQeUlkbpXduMqcBPb8BiqLWzA)

In [ ]:
import time
import requests
import jwt
from datetime import datetime
from jwt import PyJWKClient

# Helper functions for decoding and verifying JWTs
# These are made available for educational purposes and to assist with token verification in this notebook. They should not be used as a substitute for proper security practices in production environments.
def discover_jwks_uri(expected_issuer: str) -> str:
    """Discover JWKS URI from the issuer metadata endpoint."""

    print(f"⏳ Discovering JWKS URI from issuer metadata: {expected_issuer}/.well-known/openid-configuration");

    config_url = f"{expected_issuer}/.well-known/openid-configuration"
    response = requests.get(config_url, timeout=15);
    response.raise_for_status();
    jwks_uri = response.json().get("jwks_uri");
    if not jwks_uri:
        raise ValueError(f"No jwks_uri found in metadata for issuer: {expected_issuer}");

    print(f"   ☑️ JWKS URI discovered: {jwks_uri}");

    return jwks_uri

def verify_jwt_with_jwks(token: str, expected_issuer: str, expected_audience: str, label: str, scopes: list[str] = []) -> dict:
    """Verify JWT signature and registered claims using issuer JWKS."""

    print(f"⏳ Verifying JWT signature and claims for {label}...");

    jwks_uri = discover_jwks_uri(expected_issuer);
    jwk_client = PyJWKClient(jwks_uri);
    signing_key = jwk_client.get_signing_key_from_jwt(token);

    decoded = jwt.decode(
        token,
        signing_key.key,
        algorithms=["RS256", "RS384", "RS512", "ES256", "ES384", "ES512"],
        audience=expected_audience,
        issuer=expected_issuer,
    );

    print(f"✅ Signature verification passed for {label}");
    print(f"✅ Issuer matches: {expected_issuer}");
    print(f"✅ Audience matches: {expected_audience}");

    # Keep educational claim visibility in output.
    print("\nDecoded Claims Preview (unverified):");
    print(f"   iss: {decoded.get('iss')}");
    print(f"   aud: {decoded.get('aud')}");
    print(f"   sub: {decoded.get('sub')}");

    # Check expiration for readable display.
    exp = decoded.get("exp")

    if exp and exp > time.time():
        exp_time = datetime.fromtimestamp(exp)
        print(f"\n✅ Token valid until: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        raise(ValueError("Token has expired. Please obtain a new token and verify again."))

    if scopes:
        print(f"\n⏳ Checking for expected scopes: {scopes}")

        for scope in scopes:
            scope_claim = decoded.get('scp', decoded.get('scope', ''));

            if isinstance(scope_claim, str):
                scope_claim = scope_claim.split();

            if scope not in scope_claim:
                raise ValueError(f"Missing expected scope: {scope}");

            print(f"   ☑️ {scope}");

    print(f"✅ All expected scopes are present in the token for {label}.")

    return decoded

print("\n" + "=" * 80)
print("⑤ Verify User's Token")
print("=" * 80)

print(f"   Expected Audience: {PRINCIPAL_A_RESOURCE_URI}\n")

assert isinstance(token_to_verify := STATE["user"].get(
    "access_token"), str), "❌ Missing ACCESS_TOKEN. Run the previous step and try again."

try:
    decoded_access = verify_jwt_with_jwks(
        token=token_to_verify,
        expected_issuer=DOMAIN_A_ISSUER,
        expected_audience=PRINCIPAL_A_RESOURCE_URI,
        label="user access token",
        scopes=CLIENT_SCOPES
    )
except Exception as e:
    print(f"❌ Verification failed: {e}")
    raise

---

## Step 3: Exchange Access Token for ID-JAG Token

The next step is to exchange the validated user's access token for an ID-JAG token. This token represents the user's identity and can be used for cross-application access.

### The Sequence / Flow:

6. The SDK calls the Okta Org Authorization Server's token endpoint;
   - It generates a JWT bearer assertion using Agent J's private key (or uses basic auth using the Agent J's client secret).
7. After authenticating the agent, Okta exchanges the user's token and returns the ID-JAG token with the specified audience;

[![](https://mermaid.ink/img/pako:eNp1Uk1v2zAM_SsEd9kAO7Dj2HGEIYCxDsU6dD2k3WHQRbZZx0gsZbRcZA3y3yvZSdHLBH0-Pj6KlE5YmZpQYE9_B9IV3bSqYdVJDa6pwRo9dCUxZBMyzQfFtq3ag9IWihhUD0VDbn_3P9JDsfGsh51VUAx2-wob4hfij3ymygI35ed4GQXg-jxNv0gtrbS_jCUwju-iCdhYOkAi4JYsKA0_bsK74nYiSlvE4Xrtwgn4fqy2SjcET713rCrqe3g0O9JfS14_G754glXckG11c8ni51XLyYROzce8UEd3byZdY4ANtzUKywMF2BF3yh_x5PORaLfUkUThtkz1cAxrxbuwMnvDEoOJ46F79wCeNsp8cP2tuFXlnnpvPF0qhaWqdg2bQdeT9KckSSR661nqs7uTq_cfY7rrtRy12aJ4VvvenYZDrez1jd9RdtkQf3OiFsU8SUcRFCc8ooiX-SxfZItltHJLtIiSAP-hWGYzj6TzlQPzeJWfA3wdw0azPMridOVGGmdZNs8CpLq1hu-nnzZ-uPMbys_E3w?type=png)](https://mermaid.live/edit#pako:eNpdUsGK2zAQ_ZVhemnBCXYcO44oAdMtS7ds95C2h-KLbM86JrGUjuWSbsi_78hOylIhIenNm_c0ks5Y2ZpQYU-_BzIV3bW6Yd0VBqTpwVkzdCUxpBNSmGk-anZt1R61cZBHoHvIG5L1w432P-kp33rW095pyAe3e4Et8R_it3ymygE35ftoFQYgfZEkH8TRFe6bdQRW-OKmYOvoCLGCe3KgDXy5mz3k9xOxcHk022zETsHnU7XTpiH40fvEqqK-h-92T-ZjyZtny9dMcJobcq1prlV8vWmJzEzUvOeVOqb7MJkaA2y4rVE5HijAjrjTfotnX0-BbkcdFahkyVQPp1mteT-r7MFygcHE8dCjPIGnjTJvUn9qbnV5oN4Hz9ebwlJX-4btYOpJ-l0cxwX66KUwFzmT3Pcva7vbsYTa7FA960Mvu-FYa3d75X8oSzXEn0TUoVrE0SiC6ownVNEqm2fLdLkK1zKFyzAO8C-qVTr3SLJYC5hF6-wS4MtoG86zMI2StYwkStN0kQZIdessP05_bfxyl1e3-8Wl)

#### In this exercise we will...
1. Initialize an instance of the Okta SDK specifically for Agent J;
2. Send the user's access token to the Okta Org Authorization Server;
3. Receive our *first* ID-JAG!

### But first... Initialize the Okta SDK for _Agent J_

We will need to initialize a new instance of the SDK with cross-app access configuration for *Agent J* to use. But before we do that, let's walk through some of the parameters to better understand what's actually happening. 

#### `audience` vs `resource`
When initializing the SDK and using public/private key, the `audience` value provided is used to populate the `aud` claim of the JWT assertion and represents *who will validate* the assertion, which will be the authorization server that mints the ID-JAG. 

The `resource` identifies *what will be accessed* (i.e. your API). 

*Both are required but serve different roles.*

<br>

---

*NOTE*: The `resource` parameter is not set as part of the SDK initialization. Instead, it is passed as part of the token exchange call.

---

<br>

| Input Parameter | ID-JAG Claim | Explanation |
|---|---|---|
|`audience` / `issuer` (*target*)| `aud` | The "resource" authorization server URL -- the recipient of the ID-JAG.<br><br>This is set in the SDK as the *target* issuer, which can be confusing.<br><br>For this step, this is the **authorization server for Agent K.**|
|`resource`| `resource` | The Resource Server API URL.<br><br>In this step that is the resource URI you set in Agent K's configuration when allowing calls from Agent J. |
|`issuer` (*source*)| `iss` | This is the issuer of the ID-JAG. Typically the IDP. |
|`client_id`| `client_id` | The **input** value is the *Okta* client identifier for the agent. <br><br>**However**, the claim in the ID-JAG *should be an identifier recognized by the target authorization server* which may not be the same as the value sent in the request.|
|`client_secret`| n/a | This is the *Okta* issued client secret if a public/private key is not being used. |
|`private_jwk`| n/a | This is the JWK issued by *Okta* (or BYO) used by the agent to authenticate to *Okta* |



In [ ]:
print("\n⏳ Initializing Agent J's SDK...")

if "build_client_authorization" not in globals():
    raise SystemExit(
        "❌ Missing helper function. Re-run Cell 6 (SDK initialization) and then run this cell again.")

# Create OAuth2 client
# The SDK will use either private key JWT assertion or client secret auth to perform token exchange operations.
agent_j_sdk = OAuth2Client(
    configuration=OAuth2ClientConfiguration(
        # The issuer points to the Okta Org authorization server as it is the authority for issuing the ID-JAG.
        issuer=OKTA_DOMAIN,
        scope=PRINCIPAL_A_SCOPES,
        client_authorization=build_client_authorization(
            client_id=PRINCIPAL_A_ID,
            client_secret=PRINCIPAL_A_SECRET,
            private_jwk=PRINCIPAL_A_PRIVATE_JWK,
            label="Agent J"
        ),
    ))
STATE["agent_j"]["sdk"] = agent_j_sdk

if SDK_DEBUG_ENABLED:
    agent_j_sdk.listeners.add(utils.Debugger())  # type: ignore

print("✅ Agent J's SDK initialized successfully!")

### ⑥ Exchange Token
Great! Now let's start the token exchange for Agent J.

**NOTE:** *You can run this step as many times as you want so long as there is a valid access or refresh token from previous steps.*

In [ ]:
from okta_client.authfoundation import Token
from collections.abc import Sequence
from typing import Literal

# ============ HELPER FUNCTION(s) ==========
async def get_id_jag(sdk: CrossAppAccessFlow, token: str, audience: str | None, resource: Sequence[str] | None = None, scope: Sequence[str] | None = None, token_type: Literal["access_token", "id_token"] = "access_token") -> Token:
   """Helper function to obtain an ID-JAG token using the cross-app access flow.

   Args:
       sdk (CrossAppAccessFlow): The cross-app access flow instance.
       token (str): The user's access token.
       audience (str | None): The intended audience for the ID-JAG token.
       resource (Sequence[str] | None, optional): The resources for the token exchange. Defaults to None.
       scope (Sequence[str] | None, optional): The scopes for the token exchange. Defaults to None.
       token_type (Literal["access_token", "id_token"], optional): The type of token to request. Defaults to "access_token".

   Returns:
       Token: The obtained ID-JAG token.
   """
   try:

      result = await sdk.start(
         token=token,
         token_type=token_type,
         audience=audience,
         resource=resource,
         scope=scope
      )

      if result.resume_assertion_claims is not None:
        print("⚠️ Flow requires manual assertion claims. This notebook expects automatic exchange.")
        print(f"   Required claims: {result.resume_assertion_claims}")
        raise SystemExit("❌ Unable to continue automatically.")

      token_obj = sdk.context.id_jag_token if sdk.context is not None else None

      if token_obj is None:
         raise SystemExit("❌ ID-JAG token was not returned. Check token exchange configuration and retry.")

      utils.token_exchange_out(token_obj, focus="id_jag", expected_issuer=OKTA_DOMAIN)

      return token_obj

   except Exception as e:
      print(f"❌ ERROR: {e}")
      raise
# ========== END OF HELPER FUNCTION(s) ==========

print("\n" + "=" * 80)
print("⑥ Exchange user's access token for an ID-JAG token targeting Agent K.")
print("=" * 80)

# Create target
agent_j_sdk_target = CrossAppAccessTarget(
    # This is the issuer that will be receiving the ID-JAG token.
    issuer=DOMAIN_B_ISSUER
)

print(f"✅ Target created: {agent_j_sdk_target.issuer}")

# Make sure we have a valid SDK
assert isinstance(sdk := STATE["agent_j"].get(
    "sdk"), OAuth2Client), "❌ Missing or invalid SDK client for Agent J. Reinitialize the SDK and try again."

# Create cross-app flow
agent_j_auth_flow = CrossAppAccessFlow(
    client=sdk,
    target=agent_j_sdk_target
)
STATE["agent_j"]["auth_flow"] = agent_j_auth_flow

print("✅ Cross-app access flow created")

# Make sure we actually have an access token in state
assert isinstance(token := STATE["user"].get(
    "access_token"), str), "❌ Missing ACCESS_TOKEN. Run the previous step and try again."

agent_j_id_jag_token = await get_id_jag(
   sdk=agent_j_auth_flow,
   token=token,
   audience=DOMAIN_B_ISSUER,
   resource=[PRINCIPAL_B_RESOURCE_URI],
   scope=PRINCIPAL_A_SCOPES
)

# This is not absolutely necessary as the SDK caches the current ID-JAG token internally.
# However, it is helpful to store it in the state for easy access in subsequent steps for this exercise.
STATE["agent_j"]["id_jag_token"] = agent_j_id_jag_token

---

## Step 4: Verify ID-JAG Token

Before using the ID-JAG token, we *should* verify its authenticity. This step:
- Validates the token signature using Okta's public keys (JWKS)
- Checks the audience, issuer, and expiration claims
- Extracts user information from the token

***SECURITY NOTE:*** *Always verify tokens before trusting their contents.* 

This prevents:
- Tampered tokens
- Expired tokens
- Tokens meant for different audiences

[![](https://mermaid.ink/img/pako:eNpNUcFOwzAM_ZXIXEBqp6YdXZvDpAokBNK4gDigXLLW66qtTfFSNJj27zgtQySR4_i9vDj2CUpbISg44MeAXYn3janJtLoTPMzgbDe0aySRTZHJ9oZcUza96ZwopDAHUdTI_pPueDrC0gmq19dRIORtykbGNx7Q7tk6FPaTBQupxIvDXsyVeENqNl8Tw_ZucrQrZLhcet6Ei8f78Kl4EK92h93Ewa7yDm8QQE1NBcrRgAG0SK3xRzj5lDW4LbaoQbFLWA3HsDK0C0u7t6QhmDg-tOJyeNoo8-_qm6HGrPd48ODptxiwNuWuJjtwGqP0VZIkGjx61t2Zc-ISvVvbXtJiar0FtTH7A5-GvjLuUvG_KPFvkO5Y1IGK82wUAXWCI6hwsZglSSYjyStKszwJ4ItZ8XwmozxP4zSL4ySfZ-cAvsd35SzKo3iRRlGcZUkupQwAq8ZZWk2NH_t__gFIrKOA?type=png)](https://mermaid.live/edit#pako:eNpNUU1PwzAM_SuRuYDUTk07ujYHpAkkBNK4gHZAuaStV6qtTfFStDHtv-O0DJFEjj-enRf7BKWtEBTs8XPArsSHxtRkWt0JXmZwthvaAmmyJ9kbck3Z9KZzYimF2Ytljaw_6463IyydoLq4jgIhb1MWMr7xAe1erENhv5A4T4lXh72YK7FGajbHCWF7NynaLWV4d-dxU1w8PYTPy0fxZrfYTRjsKq_wBQHU1FSgHA0YQIvUGm_CyVPW4D6wRQ2KVcJqOISVoW1Y2p0lDcGE8a4VN8PDxjL_UteGGlPscO-Dp99mQGHKbU12YBpj6askSTT46Fl3Z-bELXq3tr3QYmj9AWpjdnu2hr4y7tLvPy_xb5DuuagDFefzsQioExxAhYvFLEkyGUk-UZrlSQBHRsXzmYzyPI3TLI6TfJ6dA_ge35WzKI_iRRpFcZYluZQyAKwaZ2k1jX2c_vkHhZujJA)

### Token Claims Reference

| Claim | Name | Meaning in This Notebook |
|---|---|---|
| `iss` | Issuer | Should match the expected Okta authorization server (i.e. org server) issuer for the current token. |
| `aud` | Audience | Must match the target resource or configured audience for this step. |
| `exp` | Expiration time | Token must still be valid. |
| `sub` | Subject | The entity the token references. The human identity in this flow. |
| `cid` | Client ID | Client/principal identifier (the workload or app identity). |
| `scp` | Scope | Granted scopes used for authorization decisions. |

In [ ]:
print("\n" + "=" * 80);
print("⑧ Verify ID-JAG token");
print("=" * 80);

print(f"   Expected Audience: {agent_j_sdk_target.issuer}\n");

# Make sure we have an ID-JAG to validate
assert isinstance(current_id_jag_token := STATE["agent_j"].get("id_jag_token"), Token), "❌ Invalid or missing ID-JAG token object!"

try:
    verify_jwt_with_jwks(
        token=current_id_jag_token.access_token,
        expected_issuer=OKTA_DOMAIN,
        expected_audience=DOMAIN_B_ISSUER,
        label="ID-JAG token targeting Agent K",
    )
except Exception as e:
    print(f"❌ Verification failed: {e}")
    raise

---
## Step 5: Exchange ID-JAG Token for Access Token

Now we exchange the verified ID-JAG token for an access token targeted to Agent B. 

### The Sequence / Flow:

9. Agent J calls Agent K's authorization server's token endpoint (which could be the same as Agent J).
   - The SDK generates a JWT bearer assertion for the authorization server (*or uses client_id/client_secret if configured*)
   - Uses the ID-JAG token as the assertion
10.  Returns an access token with appropriate scopes

[![](https://mermaid.ink/img/pako:eNp9Ul2PmzAQ_Cur7UsrkRRiSAKqIqFLVd2dcpWaax8qvxjYAErA6WKq9KL899pwOfVLRQLbszOzu3jPmOuCMMGOvvXU5rSuVcmqkS3YR_VGt32TEUM8IuM30yd4_PR5-wjrj5v09uFdxiv33hxqag28hbR0653DRsVRsanz-qgsnAaguitlDFNbyPb_3iP_frT8h-ks3Q62vameYEv8nfg3b2mYcgNcZq9j34Mw9sB_42BpHrQh4LqsDOidLS-BraEjRAm8P-WVakuC2_XkLv0w0qVJg8lq5TL-xYCd5ufW7kHlOXUdGL2n9kVqVROntlnSP-JDpehhyXWBieGePGyIG-WOeHbtSDQVNSQxsVumoj9NCsX7Sa4PmiV6I8dBG3uvjjbY_CL9orhW2YE6Fzw_3ypmKt-XrHtbwWD9Sggh0UUvsr3YmuxP_qp1cy3LUssKk506dPbUHwtlrqPzgrLth_jGmhpMxCIcTDA54wmTIIym4VJEkd2IZTCPZh7-sHAQT8V8tozDKPZF5AcXD5-GtP50sRAiCEQwj-cLP_AjD6mojebNOMDDHF9-AlOj4W4?type=png)](https://mermaid.live/edit#pako:eNp9Ul2PmzAQ_Cur7UsrkRTHgQRURUKXqro75So11z5UvBjYAErA6WKq9KL899pwOfVLRTK2Z2fGu_aeMdcFYYwdfeupzWldq5JVk7ZgP9Ub3fZNRgzRiIz_TJ_g8dPn7SOsP26S24d3Ga_cuDnU1Bp4C0np5juHjYqjYlPn9VFZOBGguitlDFNbpO3_vUf-_Wj5D9NZsh1se1M9wZb4O_Fv3qlhyg1wmb2OfA_mkQf-Gwen5kEbAq7LyoDeWacYtoaOEMTw_pRXqi0JbteTu-TDSE9NIiarlTvxLwbsND-Xdg8qz6nrwOg9tS9Sq5o4tYgh-SM-ZIoellwXGBvuycOGuFFui2dXToqmooZSjO2SqehPk0LxfpLrg-YUvZHjoI19V0cbbH6RflFcq-xAnQuen18VM5XvS9a9zWCwfiWlTNFFL2l7sTnZS_6qdXNNy1LLCuOdOnR21x8LZa6t84KyrYf4xpoajGUYDiYYn_GEsZgH0_lSBoFdyKUIg5mHPywsoqkMZ8toHkS-DHxx8fBpONafLhZSCiFFGIULX_iBh1TURvNmbOChjy8_AVXD4XA)

<br>

**NOTE:** *You can only run this step ONE TIME. If you want to run it again, rerun the token exchange to get a fresh ID-JAG.*

In [ ]:
from okta_client.authfoundation import Token

async def exchange_id_jag_token(sdk: CrossAppAccessFlow) -> Token:
  """Exchanges an ID-JAG token for resource access token."""
  try:

    result = await sdk.resume()

    return result

  except Exception as e:
      print(f"❌ ERROR: {e}")
      raise

print("\n" + "=" * 80)
print("⑨ Exchange ID-JAG for resource Agent K access token")
print("=" * 80)

assert isinstance(sdk := STATE["agent_j"].get("auth_flow"), CrossAppAccessFlow), "❌ Invalid or missing SDK client for Agent J. Reinitialize the SDK and try again."

agent_j_result = await exchange_id_jag_token(sdk)

utils.token_exchange_out(agent_j_result, focus="id_jag", expected_issuer=DOMAIN_B_ISSUER)

# Store for next step
STATE["agent_j"]["access_token"] = agent_j_result.access_token

---
## Step 6: Verify Agent J's Access Token

When Agent J makes the API call to Agent K, standard practice would be to validate the incoming token before giving Agent J access to any resources.

### The Sequence / Flow:
11. Agent J sends a request to Agent K using the newly minted access token in the `Authorization` header.
12. Agent K validates the token signature using the authorization server's public keys
    - Checks audience, issuer, and expiration
    - Extracts scope and user information

[![](https://mermaid.ink/img/pako:eNp9Ut1vmzAQ_1es28smkdSGwMCaIqH2pavSTW3Wh4oXY64UBTAzZksb5X-fjZt2k7Yh4Y_7ffjOvgNIVSFwGPH7hL3Ei0bUWnRFT-wnJqP6qStRE8Z8yI-l2pPtzbfbLbn4sskvrz-Veu3-87bB3pAzktdu_uxiXjEIbRrZDMKGc0bEeKJ4GPuq6P_v7flX_7AM3yyv_rAsjEZpiK5L8Z4GJE0CwsLMDh8cVphrZZCoH7bEnHFya3AgCSf510siRduSn2dESInjSIzaYe81OVus13nIyY27tdFY7HS2E-ResH0TqMH4hdWGL9o71M3D01_IPnE3QQC1birgRk8YQIe6E24LB1diAeYROyyA26XGatovKqF3C6lapQsIPMeFNvaNHW22-U16J3QjyhZHBx5eHhhKIXe1VpNNY7Z-F0VRAQ49Fv3R5mRv_F6p7pSWpdaPwB9EO9rdNFTCnNroNaptNajPrakBnlI2mwA_wB54uEqXNFzRjMVpyGhswSfgq9WSZRmNozhLP9KIJccAnudT6TKlcZaFNEnShFpSAFg1RumNb-W5o4-_AM4i5dA?type=png)](https://mermaid.live/edit#pako:eNp9Ut9vmzAQ_les28smkdSGQMCaIqH2pavSTW3Wh4kXY64UBTAzZksX5X-fjZt1k7bZ8q-77_t8Z98RpKoQOIz4dcJe4lUjai26oie2icmofupK1IQxb_JzqQ5kd_f5fkeuPm7z69v3pd64cdk22BtyQfLarR-czTMGoU0jm0FYc86IGM8Q78a-KnrX_6fuGTf_EA1fRW_-FDUapSG6LsVbGpA0CQgLMzu9c77C3CqDRH2zSeaMk3uDA0k4yT9dEynalny_IEJKHEdi1B57z8nZYrPJQ07u3LuNxvrOdztC7gm7V4IajN9YbvjCfUDdPD7_BewDdwsEUOumAm70hAF0qDvhjnB0KRZgnrDDArjdaqymw6ISer-QqlW6gMBjnGlrf9nBZpnfqA9CN6JscXTO48sXQynkvtZqsmHM0m-iKCrAeU9Ff7Ix2Rf_olR3DstC6yfgj6Id7WkaKmHOhfTLqm02qC-tqAG-ztazCPAjHICHq3RJwxXNWJyGjMYsgGfgq9WSZRmNozhL1zRiySmAH_OtdJnSOMtCmiRpQi0oAKwao_TWF_Nc06ef_rXmqA)

#### Important:
The resource server (*your API*) should perform this verification on every request to ensure:
- Token is valid and not tampered with
- Token is not expired
- Token is intended for this resource server (*audience check*)
- User has required permissions (*scope check*)

In [ ]:
print("\n" + "=" * 80)
print("①⓪ Verify Access Token")
print("=" * 80)

if "verify_jwt_with_jwks" not in globals():
    raise SystemExit("❌ Missing verification helper. Run Step 3 first.")

assert isinstance(access_token := STATE["agent_j"].get("access_token"), str), "❌ Missing or invalid access token for Agent J. Run previous step before continuing."

verify_jwt_with_jwks(
    token=access_token,
    expected_issuer=DOMAIN_B_ISSUER,
    expected_audience=PRINCIPAL_B_RESOURCE_URI,
    label="Agent K access token",
    scopes=PRINCIPAL_A_SCOPES
)

---

## Step 7: Exchange Verified Access Token for an ID-JAG Token

Next we need to exchange the now verified incoming access token (*targeting Agent K*) for an ID-JAG targeting the *MCP Resource*.

### The Sequence / Flow:

13. Agent K sends a request to the IDP with the access token sent by Agent J in the request;
    - The SDK generates a JWT bearer assertion for the IDP authorization server (*or uses client_id/client_secret if configured*)
    - Uses the access token as the assertion
14. After authenticating the agent, Okta exchanges the access token and returns the ID-JAG token with the specified audience;

[![](https://mermaid.ink/img/pako:eNplUmFvmzAQ_Ssn78smQYQhEIqmSGiZqm5KIzVdP1R8MXAlKMHODjNljfLfe4ZmqjQjLN-9d-8e-M6iMjWKTPT4e0Bd4apVDamu0MBLDdbooSuRQEZTatqPimxbtUelLWzyLageNnurIB_s7hW2SH-QPvJLc4LHh1_bR1ht1vnd_deSlu7NG2SFn-74v3AeOt13ygSjrgvNjyWsLFBTqs8yjTwIPJBxyNsXBxb23lgEwyacO4-VMthaPMIig1u0oDTcrfwf-e2k6lYe-sslkzP4fqp2SjcIqqqw78GaPWp4MeRcTmVgFTVoW93AA_ZmoAqnvizgs47rd2W6ageO1oUnGmprkVka0BMdUqdcKM7OSSHsDjssRMZHwno4-bWivV-Zg6FCeBPHpdZ8aY42ynwofVLUqvKAvQPP7zcgSlXtGzIDOxilP0VRVAiHXgp9YU_8u5-N6a62mNrsRPaiDj1Hw7FW9joX_7LE34P0jUWtyKSM41FFZGdx4jiRs3m0SBfzNJKxnN8knvjL6TSdBYmMUhlGQSzjaH7xxOvYOJjdJEkog3CeBGkiw4ArsG6tofU0n-OYXt4AhvDUMw?type=png)](https://mermaid.live/edit#pako:eNplUmFr2zAQ_SuH9mUDO1iW47hmBMwySjfSQNPtw_AXWb46JrGUneWRNeS_V7KTUZiMhe7eu3fP1p2ZMjWynPX4e0CtcNXKhmRXanBLDtbooauQgIspNe1HSbZV7VFqC5tiC7KHzd5KKAa7e4Ut0h-k9_zKnOD56cf2GVabdfHw-LmipX-LBp3Cd3_8X7iIve6VMsGo61JfH0uoLFBTyY88EwFEAfB57LZPHizto7EIxhnxDgOnlsPW4hEWOdyjBanhYRV-K-4nZb-KOFwuHTmHrye1k7pBkEph34M1e9TwYsg7ncrASmrQtrqBJ-zNQAqnvk4gdDq-343pqz042mcBa6itWW5pwIB1SJ30ITt7JyWzO-ywZLk7EtbDKawl7UNlDoZKFkwcn1q7i_O0UeZd6U9JrawO2HvwfL0FVkm1b8gMzsEo_UEIUTKPXkp9cZ7cL_9lTHez5ajNjuUv8tC7aDjW0t5m41-W3PcgfXGiluWcJ4tRheVndnJxymeJWGSLJBN8zpO7NGB_XTrLZlHKRcZjEc35XCSXgL2OjaPZXZrGPIqTNMpSHkeuAuvWGlpPMzqO6uUNkRHVyA)

### But first... Initialize the Okta SDK for _Agent K_

We will need to initialize a new instance of the SDK with cross-app access configuration for *Agent K* to use. We did this once before but now we are calling a differnet resource so let's walk through some of the parameters to better understand what's actually happening. 

<br>

| Input Parameter | ID-JAG Claim | Explanation |
|---|---|---|
|`audience` / `issuer` (*target*)| `aud` | The *resources* authorization server URL -- the recipient of the ID-JAG. <br><br>This is set in the SDK as the *target* issuer which can be confusing.<br><br>For this step that is the authorization server for the **MCP Resource**.|
|`resource`| `resource` | The Resource Server API URL. <br><br>For this step, this value was set when configuring Agent K's connection to the **MCP Resource**. |
|`issuer` (*source*)| `iss` | This is the issuer of the ID-JAG. Typically the IDP. |
|`client_id`| `client_id` | The **input** value is the *Okta* client identifier for Agent K.<br><br>**However**, the claim in the ID-JAG *should be an identifier recognized by the target authorization server* which may not be the same as the value sent in the request.|
|`client_secret`| n/a | This is the *Okta* issued client secret if a public/private key is not being used. |
|`private_jwk`| n/a | This is the JWK issued by *Okta* (or BYO) used by the agent to authenticate to *Okta* |



In [ ]:
print("\n⏳ Initializing Agent K's SDK...")

if "build_client_authorization" not in globals():
    raise SystemExit(
        "❌ Missing helper function. Re-run SDK initialization for the user and then run this cell again.")

# Create OAuth2 client
agent_k_sdk = OAuth2Client(
    configuration=OAuth2ClientConfiguration(
        # The issuer points to the Okta Org authorization server as it is the authority for issuing the ID-JAG.
        issuer=OKTA_DOMAIN,
        scope=PRINCIPAL_B_SCOPES,
        client_authorization=build_client_authorization(
            client_id=PRINCIPAL_B_ID,
            client_secret=PRINCIPAL_B_SECRET,
            private_jwk=PRINCIPAL_B_PRIVATE_JWK,
            label="Agent K"
        ),
    ))

STATE["agent_k"]["sdk"] = agent_k_sdk

if SDK_DEBUG_ENABLED:
    agent_k_sdk.listeners.add(utils.Debugger())  # type: ignore

print("✅ Agent K's SDK initialized successfully!")

### ①③ Exchange Token
Great! Now let's start the token exchange for Agent K.

**NOTE:** *You can run this step as many times as you want so long as there is a valid access or refresh token from previous steps.*

In [ ]:
print("\n" + "=" * 80)
print("①③ Exchange the incoming access token for an ID-JAG token targeting the Resource MCP.")
print("=" * 80)

assert isinstance(access_token := STATE["agent_j"].get("access_token"), str), "❌ Missing or invalid access token for Agent J. Run the previous step before continuing."

assert isinstance(sdk := STATE["agent_k"].get("sdk"), OAuth2Client), "❌ Missing or invalid SDK for Agent K. Run the previous steps to initialize the SDK."

# Create target
agent_k_sdk_target = CrossAppAccessTarget(
   # This is the issuer that will be receiving the ID-JAG token.
    issuer=RESOURCE_ISSUER
)

print(f"✅ Target created: {agent_k_sdk_target.issuer}")

# Create cross-app flow
agent_k_auth_flow = CrossAppAccessFlow(
    client=sdk,
    target=agent_k_sdk_target
)

# Update the global state with the newly created cross-app access flow for Agent K.
STATE["agent_k"]["auth_flow"] = agent_k_auth_flow

print("✅ Cross-app access flow created")

agent_k_id_jag_token = await get_id_jag(
   sdk=agent_k_auth_flow,
   token=access_token,
   audience=RESOURCE_ISSUER,
   scope=PRINCIPAL_B_SCOPES
)

# Update the global state with the newly obtained ID-JAG token for Agent K.
STATE["agent_k"]["id_jag_token"] = agent_k_id_jag_token

---

## Step 8: Verify ID-JAG Token

Before using the ID-JAG token, we *should* verify its authenticity. *This should be familiar as we have already done it once!* 

This step:
- Validates the token signature using Okta's public keys (JWKS)
- Checks the audience, issuer, and expiration claims
- Extracts user information from the token

***SECURITY NOTE:*** *Always verify tokens before trusting their contents.* 

This prevents:
- Tampered tokens
- Expired tokens
- Tokens meant for different audiences

[![](https://mermaid.ink/img/pako:eNpNUl1rwjAU_Svh7mWDVtpUY5sHQSaMbbiXDR9GXtL2Wou26a4p6MT_vqSdYwkk9-Pck5ObXKAwJYKEI3712Ba4qnVFulEtc0P31rR9kyOxeDaGxrXTZOui7nRr2ZIzfWTLCp39qlo3LWFhGVX5fRS4QuGWmD_4hLJvxiI74NYys3Wlkr1b7Fgq2Qap3p5HkOnsaCi75OFi4XFjnj2vwpflE_swe2xHDLalN9wGAVRUlyAt9RhAg9Ro78LFq1Zgd9igAulMwrI_haWmfViYgyEFwYjxobVriYcNNP9KN5pqnR_w6JOX335Arot9RaZ3MgbquyRJFPjsVbVXp8l16dOY5ibLQasdyK0-HJ3Xd6W2t67_RcndBunRkVqQseDZwALyAieQXIgJn0azSERznqZTEcAZZBgn80k8S0SaCc6T6TTi1wC-h4PjSZxFYpZmXGQiSSPBA8CytobW4-sPn-D6AziGpN4?type=png)](https://mermaid.live/edit#pako:eNpNklFr2zAQx7-KuL2sYAdbjlVHjIBZoHQjHbRZH4ZeZPvimsSWd5EhXch3r2S1ZQJJp7ufTn-ddIHaNAgSTvh3wqHGTadb0r0amGt6smaY-gqJpXlwhbEyZ7Z7_P20Y5tf2_L-4VtFa9_LFgfLfnpTDQHFoQnGqMl2dTdqR5Sc6RN7pz2pLGFtGbXV1yRyhwk3pPzGB5R9MBbZEfeWmb3bKtmTxZEVkj0jdfvXAJnRBkPZksfrtedCnN1v4h_lHduZAw6BmUXNE0TQUteAtDRhBD1Sr_0SLl61AvuCPSqQziRspnPcaDrEtTkaUhAFxru2rowem9P8t_VZU6erI5588PJeQ6h0fWjJTE7GnPpLlmUKfPSqhqvT5Kr0x5j-Q5ZD2xeQe308udU0Ntp-vNSnl9xtkL67pBZkKng-ZwF5gTNILsSCL5M8EcktL4qliOAVZJxmt4s0z0SxEpxny2XCrxH8mw9OF-kqEXmx4mIlsiIRPAJsOmtoG37M_HGub1hutJY)

In [ ]:
print("\n" + "=" * 80);
print("①⑤ Verify ID-JAG token");
print("=" * 80);

assert isinstance(current_id_jag_token := STATE["agent_k"].get("id_jag_token"), Token), "❌ Missing or invalid ID-JAG token for Agent K. Run the previous step before continuing."

print(f"   Expected Audience: {RESOURCE_ISSUER}\n");

try:
    verify_jwt_with_jwks(
        token=current_id_jag_token.access_token,
        expected_issuer=OKTA_DOMAIN,
        expected_audience=RESOURCE_ISSUER,
        label="ID-JAG token targeting Resource",
    )
except Exception as e:
    print(f"❌ Verification failed: {e}")
    raise

---
## Step 9: Exchange ID-JAG Token for Resource Access Token

Now we exchange the verified ID-JAG token for an access token targeting the MCP Resource.

### The Sequence / Flow:

16. Agent K calls the Resource's authorization server's token endpoint.
   - The SDK generates a JWT bearer assertion for the authorization server (*or uses client_id/client_secret if configured*)
   - Uses the ID-JAG token as the assertion
17.  Returns an access token with appropriate scopes

[![](https://mermaid.ink/img/pako:eNqNUl2PmzAQ_Cur7UsrwSmGED5URUJNVV2rXKXkeg8VL8bsEZSA08VU6UX577XhUl3Ve6gRyJ6dnR3We0alK8IMe_oxUKdo1ciaZVt0YJccjO6GtiQGsZig6VvqE9xvvm3vYfV1nd_evS956d68ps7AF7ediEfJplHNUVo4D0D28EyZwtRV_yO7oV4PrOh13U2-HYUHs3uCLfFP4hfq9jFMygDX5dsgijyYeSBE-s4FCnOnDYG2KdZeBltDR0gz-HhSO9nVBLcr_3P-aaLmgb9c2mL_hOFRM1w9glSK-h6M3lM3-XDL5vk23RXJ_yIUxtpED2tuKswMD-RhS9xKd8SzUyjQ7KilAjO7ZaqGk19J3vtKHzQX6E0cB63tZTraKPMi9UFyI8sD9S54fm45llLta9aDbdMo_SYMwwJd9FJ0F-vJtve71u3VlqXWO8we5aG3p-FYSXOdlz8o278h_mBFDWaBmMWjCmZnPGEmQnEj0iRKwngRxfMomXv4y8LCwnGapItkJiIRh8HFw6ex8OxmEcQLkUSRCKJ5EAeRh1Q1RvN6mttxfC-_AT4Q3H4?type=png)](https://mermaid.live/edit#pako:eNqNUl2PmzAQ_CvW9qWV4BRDHD5URUJNVV2rXKXk2oeKFwN7BCXgdDFVelH-e9eQVFfdS41A9uzs7LDeM5SmQkihx58DdiWuGl2TbvNO8NKDNd3QFkhCLiZo-hbmJB4337aPYvV1nd0_vC9o6d6sxs6KL247EY-abFM2R81wFgjdiytlCmNX_Y_sBnszUImT7mvlTbYdpQe7exZbpF9IL_T5sYSlFVQXbwOlPDHzhJTJOxfI7YOxKAynsMFUbC0eRZKKj6dyp7saxf3K_5x9mqhZ4C-XXOxVWDwZEjeXQpcl9r2wZo9Xs25xns_prkj2DyG3bBM8qKmpILU0oActUqvdEc5OIQe7wxZzSHlLWA0nv9K090tzMJSDN3EctObrdLRR5kXqd02NLg7Yu-D52nQodLmvyQzcplH6TRiGObjoJe8u7Inb-8OY9maLqfUO0id96Pk0HCttbxPzFyX-G6QPLGohDeQsHFUgPcMJUhnKO5nEKg6jhYrmKp578JthyXCUxMkinkklozC4ePA8Fp7dLYJoIWOlZKDmQRQoD7BqrKH1NLnjAF_-AFGc3UQ)

In [ ]:
print("\n" + "=" * 80)
print("①⑥ Exchange ID-JAG for resource access token")
print("=" * 80)

if "exchange_id_jag_token" not in globals():
    raise SystemExit("❌ Missing helper function. Re-run Step 5 and then run this cell again.")

assert isinstance(sdk := STATE["agent_k"].get("auth_flow"), CrossAppAccessFlow), "❌ Missing or invalid cross-app SDK flow for Agent K."

agent_k_result = await exchange_id_jag_token(sdk)

# Store for next step
STATE["agent_k"]["access_token"] = agent_k_result.access_token

---
## Step 10: Verify Resource Access Token

The final step would be to call the resource, which *should* then validate the incoming access token.

For this exercise we will do that manually.

### The Sequence / Flow:
18. Agent K sends a request to the Resource using the newly minted access token in the `Authorization` header.
19. The Resource validates the token signature using the authorization server's public keys
    - Checks audience, issuer, and expiration
    - Extracts scope and user information

[![](https://mermaid.ink/img/pako:eNqNUk2P0zAQ_SsjcwGRLrXTNGmEKkW7lxXqgrJlDygXJxnSqElcbAcKVf8747i7KogDjmLZ8968-fCcWKVqZCkz-G3EocK7VjZa9sUAtORo1TD2JWrgiTf5vVRH2OafH7dw93GT3T-8L_Xa_VmDg4UP7uiJB6ltW7UHSeZMgDRwoXgYh_p_ZHM0atQV_ls3d7LPlCtd-qzGyoJuytdhEkAcBsDnqzcOKOyDsgjqO9WWiQDyFB4tHghPIft0D7ey6-DHO5BVhcaAVXscvF8mZrP1mvi5a5mxhL1Eh7eQeYetd4DLyr3Lk-zaWlLcP2X_Yu3kUHcI2utfw4RnIiXEHNRg0CVEpbKANbqtWWr1iAHrUffSXdnJ-RbM7rDHgqV01FiPx1kt9X5WqU7pggWe40wbGgVHm2SuXJ-kbmXZoXHg6fJgrJTVvtFqpFZP0q_CMCyYQ8_FcKac6HG-KNU_p0XUZsfSr7IzdBsPrg-XaXuxaqoG9S2JWpYKHkWTCktP7MhSHvIbvkqiJIyXUbyIkkXAfpKZkzleJatlMucRj0NxDtivKfD8ZiniJU-iiItoIWJBeli3VumNn_pp-M-_AeGq7rg?type=png)](https://mermaid.live/edit#pako:eNqNU0uPmzAQ_isj99KqZBtDCARVkdDuZVVlW5F0DxUXA1OCApjapk03yn_vGGdXabWHGmHZ8z1m_DqxUlbIEqbxx4h9iXeNqJXo8h6oidHIfuwKVMBjF3J9IY-wy75ud3D3eZPeP3ws1Nr-aY29gU926IiDUKYpm0FQOPVBaLhQXoPT7UQYzf4Jtqh-onIs7Kv_SZ6hlqMq8fXsmfV-plz50mcUlgZUXbwNYg-iwAM-X72zQG4epEGQVArV50GWwNbgQHgC6Zd7uBVtC78-gChL1BqMPGDvdKk_m63XxM_sxmpD2Et2eA-pE-ycAC4tc5JH0TaVoLx_2_7D2ou-ahGU87-GCU_9hBA9yF6jLYiWyjxWq6ZiiVEjeqxD1Qk7ZSerzZnZY4c5S2iosBqPs0qow6yUrVQ58xzHhjZ0YSxtsrmSPgrViKJFbcHT5cBYIcpDreRIWz1ZvwmCIGcWPef9mWqiw_kmZfdcFlHrPUu-i1bTbBzsPlzu5EtU0WpQ3ZKpYYnPF9HkwpITO7KEB_yGr-IwDqJlGC3CeOGx3xTmFI5W8WoZz3nIo8A_e-xpSjy_WfrRksdhyP1w4Ud-6DGsGiPVxr2N6Ymc_wAT9_uA)

#### Important:
The resource server (*your API*) should perform this verification on every request to ensure:
- Token is valid and not tampered with
- Token is not expired
- Token is intended for this resource server (*audience check*)
- User has required permissions (*scope check*)

In [ ]:
print("\n" + "=" * 80)
print("①⑨ Verify Access Token")
print("=" * 80)

if "verify_jwt_with_jwks" not in globals():
    raise SystemExit("❌ Missing verification helper. Run Step 3 first.")

assert isinstance(token_to_verify := STATE["agent_k"].get("access_token"), str), "❌ Missing or invalid access token for Agent K."

decoded_access = verify_jwt_with_jwks(
    token=token_to_verify,
    expected_issuer=RESOURCE_ISSUER,
    expected_audience=RESOURCE_SERVER_AUDIENCE,
    label="Resource access token",
    scopes=PRINCIPAL_B_SCOPES
)

##### Congratulations on successful completion of the multi-agent cross app access flow and securing your AI Agents with Okta!

---

## Resources

- [Okta AI SDK Documentation](https://github.com/okta/okta-client-python/tree/main)
- [ID-JAG - Identity Assertion Authorization Grant](https://datatracker.ietf.org/doc/draft-ietf-oauth-identity-assertion-authz-grant/)

---
